In [4]:
# Pima Indians Diabetes Statistical Analysis

#Dataset: Pima Indians Diabetes Dataset (768 rows, 9 columns)

#This project performs:
#- Data loading with exception handling
#- Data exploration
#- Missing value handling
#- Feature engineering
#- Risk analysis
#- Statistical calculations
#- Report generation

In [5]:
import pandas as pd

In [6]:
def load_data():
    try:
        df = pd.read_csv("data/diabetes.csv")
        print("File loaded successfully!\n")
        return df

    except FileNotFoundError:
        print("Error: File not found.")

    except PermissionError:
        print("Error: Permission denied.")

    except Exception as e:
        print("Unexpected error:", e)

In [7]:
df = load_data()

File loaded successfully!



In [8]:
if df is not None:
    print("First 10 rows:\n", df.head(10))
    print("\nLast 5 rows:\n", df.tail(5))

    print("\nData Info:")
    df.info()

    print("\nPatients with Glucose > 140 OR Age > 50:\n")

    count = 0
    for i in range(len(df)):
        if df.loc[i, 'Glucose'] > 140 or df.loc[i, 'Age'] > 50:
            print(df.loc[i])
            count += 1
        if count == 10:
            break

First 10 rows:
    Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   
5            5      116             74              0        0  25.6   
6            3       78             50             32       88  31.0   
7           10      115              0              0        0  35.3   
8            2      197             70             45      543  30.5   
9            8      125             96              0        0   0.0   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3              

In [9]:
def handle_missing(df):
    columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

    for col in columns:
        df[col] = df[col].astype(float)

    for col in columns:
        median = df[col].median()

        for i in range(len(df)):
            if df.loc[i, col] == 0:
                df.loc[i, col] = median

    print("Missing values replaced with median.\n")
    return df

df = handle_missing(df)

Missing values replaced with median.



In [10]:
def create_features(df):

    # BMI_Glucose_Index
    df['BMI_Glucose_Index'] = (df['BMI'] * df['Glucose']) / 100

    # Insulin_Glucose_Ratio (safe division)
    ratios = []
    for i in range(len(df)):
        if df.loc[i, 'Glucose'] != 0:
            ratios.append(df.loc[i, 'Insulin'] / df.loc[i, 'Glucose'])
        else:
            ratios.append(0)

    df['Insulin_Glucose_Ratio'] = ratios

    # Age Risk Level
    risk_levels = []
    for age in df['Age']:
        if age < 30:
            risk_levels.append("Low")
        elif age < 50:
            risk_levels.append("Medium")
        else:
            risk_levels.append("High")

    df['Age_Risk_Level'] = risk_levels

    print("New features created.\n")
    return df

df = create_features(df)

New features created.



In [11]:
def add_high_risk_flag(df):
    flags = []

    for i in range(len(df)):
        if (df.loc[i, 'Glucose'] > 140 or
            df.loc[i, 'BMI'] > 30 or
            df.loc[i, 'Age'] > 50):
            flags.append(1)
        else:
            flags.append(0)

    df['High_Risk'] = flags

    diabetic = 0
    non_diabetic = 0

    for i in range(len(df)):
        if df.loc[i, 'High_Risk'] == 1:
            if df.loc[i, 'Outcome'] == 1:
                diabetic += 1
            else:
                non_diabetic += 1

    print("High-risk diabetic:", diabetic)
    print("High-risk non-diabetic:", non_diabetic)

    return df

df = add_high_risk_flag(df)

High-risk diabetic: 241
High-risk non-diabetic: 304


In [12]:
# filter() + map()
filtered = list(filter(lambda x: x > 0, df['Insulin']))
ratios_map = list(map(lambda x: x / 100, filtered))

# Manual loop
ratios_manual = []
for i in range(len(df)):
    if df.loc[i, 'Insulin'] > 0:
        ratios_manual.append(df.loc[i, 'Insulin'] / df.loc[i, 'Glucose'])

print("Map ratios:", ratios_map[:5])
print("Manual ratios:", ratios_manual[:5])

Map ratios: [0.305, 0.305, 0.305, 0.94, 1.68]
Manual ratios: [np.float64(0.20608108108108109), np.float64(0.3588235294117647), np.float64(0.16666666666666666), np.float64(1.0561797752808988), np.float64(1.2262773722627738)]


In [13]:
def calculate_total_risk(df, index=0, total=0):
    if index == len(df):
        return total

    return calculate_total_risk(
        df,
        index + 1,
        total + df.loc[index, 'BMI_Glucose_Index']
    )

# Recursive
recursive_total = calculate_total_risk(df)

# Loop comparison
loop_total = 0
for i in range(len(df)):
    loop_total += df.loc[i, 'BMI_Glucose_Index']

print("Recursive Total:", recursive_total)
print("Loop Total:", loop_total)

Recursive Total: 30690.963000000003
Loop Total: 30690.963000000003


In [14]:
cols = ['Glucose', 'BMI', 'Age', 'Insulin']

print("Pandas Statistics:\n")
print(df[cols].describe())

# Manual stats
for col in cols:
    data = list(df[col])

    total = sum(data)
    mean = total / len(data)

    sorted_data = sorted(data)
    n = len(data)

    if n % 2 == 0:
        median = (sorted_data[n//2] + sorted_data[n//2 - 1]) / 2
    else:
        median = sorted_data[n//2]

    minimum = min(data)
    maximum = max(data)

    freq = {}
    for val in data:
        freq[val] = freq.get(val, 0) + 1

    mode = max(freq, key=freq.get)

    print(f"\n{col}:")
    print("Mean:", mean, "Median:", median, "Mode:", mode)
    print("Min:", minimum, "Max:", maximum)

Pandas Statistics:

          Glucose         BMI         Age     Insulin
count  768.000000  768.000000  768.000000  768.000000
mean   121.656250   32.450911   33.240885   94.652344
std     30.438286    6.875366   11.760232  105.547598
min     44.000000   18.200000   21.000000   14.000000
25%     99.750000   27.500000   24.000000   30.500000
50%    117.000000   32.000000   29.000000   31.250000
75%    140.250000   36.600000   41.000000  127.250000
max    199.000000   67.100000   81.000000  846.000000

Glucose:
Mean: 121.65625 Median: 117.0 Mode: 100.0
Min: 44.0 Max: 199.0

BMI:
Mean: 32.45091145833333 Median: 32.0 Mode: 32.0
Min: 18.2 Max: 67.1

Age:
Mean: 33.240885416666664 Median: 29.0 Mode: 22
Min: 21 Max: 81

Insulin:
Mean: 94.65234375 Median: 31.25 Mode: 30.5
Min: 14.0 Max: 846.0


In [15]:
groups = {}

for outcome in set(df['Outcome']):
    groups[outcome] = df[df['Outcome'] == outcome]

for key, group in groups.items():
    print(f"\nOutcome {key} stats:")
    print(group[['Glucose', 'BMI']].mean())


Outcome 0 stats:
Glucose    110.6820
BMI         30.8802
dtype: float64

Outcome 1 stats:
Glucose    142.130597
BMI         35.381343
dtype: float64


In [18]:
try:
    with open("diabetes_analysis_report.txt", "w", encoding="utf-8") as f:

        f.write("PIMA INDIANS DIABETES ANALYSIS REPORT\n")
        f.write("="*60 + "\n\n")

        f.write("1. DATASET INFORMATION\n")
        f.write("-"*40 + "\n")
        f.write(f"Total Patients: {len(df)}\n")
        f.write(f"Columns: {list(df.columns)}\n\n")

        f.write("2. STATISTICAL SUMMARY\n")
        f.write("-"*40 + "\n")
        desc = df[['Glucose','BMI','Age','Insulin']].describe()
        f.write(str(desc) + "\n\n")

        f.write("3. MEAN VALUES\n")
        f.write("-"*40 + "\n")
        means = df[['Glucose','BMI','Age','Insulin']].mean()
        for col, val in means.items():
            f.write(f"{col}: {val:.2f}\n")
        f.write("\n")

        f.write("4. MIN & MAX VALUES\n")
        f.write("-"*40 + "\n")
        for col in ['Glucose','BMI','Age','Insulin']:
            f.write(f"{col} → Min: {df[col].min()} | Max: {df[col].max()}\n")
        f.write("\n")

        f.write("5. MEDIAN VALUES\n")
        f.write("-"*40 + "\n")
        for col in ['Glucose','BMI','Age','Insulin']:
            f.write(f"{col}: {df[col].median():.2f}\n")
        f.write("\n")

        f.write("6. HIGH RISK ANALYSIS\n")
        f.write("-"*40 + "\n")
        high_risk = df['High_Risk'].sum()
        total = len(df)

        f.write(f"Total High Risk Patients: {high_risk}\n")
        f.write(f"Percentage High Risk: {(high_risk/total)*100:.2f}%\n\n")

        diabetic_hr = df[(df['High_Risk']==1) & (df['Outcome']==1)].shape[0]
        non_diabetic_hr = df[(df['High_Risk']==1) & (df['Outcome']==0)].shape[0]

        f.write(f"High Risk Diabetic: {diabetic_hr}\n")
        f.write(f"High Risk Non-Diabetic: {non_diabetic_hr}\n\n")

        f.write("7. OUTCOME ANALYSIS\n")
        f.write("-"*40 + "\n")
        diabetic = df[df['Outcome']==1].shape[0]
        non_diabetic = df[df['Outcome']==0].shape[0]

        f.write(f"Diabetic Patients: {diabetic}\n")
        f.write(f"Non-Diabetic Patients: {non_diabetic}\n\n")

        f.write("8. CORRELATION INSIGHTS\n")
        f.write("-"*40 + "\n")
        corr = df[['Glucose','BMI','Age','Insulin','Outcome']].corr()
        f.write(str(corr) + "\n\n")

        f.write("9. KEY OBSERVATIONS\n")
        f.write("-"*40 + "\n")
        f.write("Higher glucose levels strongly relate to diabetes.\n")
        f.write("BMI above 30 significantly increases risk.\n")
        f.write("Age above 50 shows higher diabetes probability.\n")
        f.write("Insulin values vary widely among patients.\n\n")

        f.write("10. CONCLUSION\n")
        f.write("-"*40 + "\n")
        f.write("Glucose, BMI, and Age are the most important indicators of diabetes risk.\n")
        f.write("High-risk patients are more likely to be diabetic.\n")
        f.write("Early detection using these features can help prevent complications.\n\n")

        f.write("Report Generated Successfully.\n")

    print("Detailed report saved successfully!")

except Exception as e:
    print("Error writing report:", e)

Detailed report saved successfully!
